# Extracting Structure from Chaos: Insurance Claims with LangGraph + Snowflake Cortex

In this hands-on lab, you'll build a production-grade pipeline to extract structured data from unstructured insurance adjuster notes using **LangGraph** for orchestration and **Snowflake Cortex** for LLM inference.

## What You'll Build

```
FEMA Flood Claims (Parquet)          Your Extraction Pipeline
========================          ========================
                                  
 Structured Data (2.7M rows)       LangGraph StateGraph
   |                                ┌──────────┐
   v                                │ EXTRACT  │ ← Cortex COMPLETE
 Generate Adjuster Notes            └────┬─────┘
   (Cortex COMPLETE)                     │
   |                                ┌────v─────┐
   v                                │ VALIDATE │ ← Pydantic
                                    └────┬─────┘
 Unstructured Text                       │
 "Inspected property at              ┌───┴───┐
  123 Main St. Water depth          valid  invalid
  reached 36 inches..."              │       │
   |                            ┌───v──┐ ┌──v──┐
   v                            │FINAL │ │ FIX │→ retry
                                └──────┘ └─────┘
 Structured Data (recovered)
   → Compare to ground truth
   → Feed into Eval Lab
```

## Learning Objectives

1. Load and explore the FEMA NFIP flood claims dataset in Snowflake
2. Use Snowflake Cortex to generate realistic unstructured text from structured data
3. Design extraction schemas using Pydantic models with validation
4. Build a LangGraph pipeline with extract → validate → fix → finalize nodes
5. Run batch extraction with checkpointing and restartability
6. Measure and optimize cost using query tags and QUERY_HISTORY
7. Evaluate performance across warehouse sizes and processing strategies
8. Plan scaling strategies for the full 2.7M row dataset

## Prerequisites

Before starting, confirm you have:

- [ ] A Snowflake account with **Cortex LLM access** (check with your admin)
- [ ] A role with the `CORTEX_USER` database role granted
- [ ] This notebook running in **Snowflake Notebooks with Container Runtime** enabled
  - Container Runtime is required because LangGraph is not available in the Anaconda channel
  - When creating your notebook, select "Run on container" under Advanced Settings
- [ ] The `FimaNfipClaimsV2.parquet` file uploaded to your notebook's stage files

## Important Notes

- **Restartable**: All intermediate results are saved to Snowflake tables. If your kernel restarts, just re-run Section 1 (setup) and skip to where you left off.
- **Interactive**: Cells marked with **EXERCISE**, **CHALLENGE**, or **EXPERIMENT** require you to write or modify code. Don't just click Enter!

---
# Section 1: Exploring the FEMA NFIP Data

The [FEMA National Flood Insurance Program (NFIP)](https://www.fema.gov/about/openfema/data-sets#nfip) dataset contains over **2.7 million flood insurance claims** across the United States. Each row includes structured fields like damage amounts, property details, geographic info, and flood characteristics.

Our plan:
1. Explore the data to understand what we're working with
2. Create a curated sample for the lab exercises


### EXERCISE: Explore the Data

Before we start generating text and extracting data, you need to understand what you're working with. Write SQL queries to answer these questions:

1. What are the **top 10 states** by claim count?
2. What is the **average building damage amount** grouped by flood event?
3. What are the **distinct flood zone** values and how many claims are in each?
4. What is the **range of water depth** values (min, max, avg)?

Understanding these distributions will help you evaluate your extraction pipeline's accuracy later.

In [ ]:
-- EXERCISE: Write your exploration queries here.
-- Hint: Use GROUP BY, COUNT(*), AVG(), MIN(), MAX()

-- 1. Top 10 states by claim count


-- 2. Average building damage by flood event (top 10 events)


-- 3. Flood zone distribution


-- 4. Water depth statistics


In [ ]:
# Snowpark DataFrames give us a complementary Python view of the data
df = session.table("RAW_CLAIMS")
print(f"Total rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.describe().show()

In [ ]:
-- Create a curated sample for the lab.
-- We filter for rows where key fields are non-null -- this ensures:
-- 1. Our synthetic notes will have rich content to generate from
-- 2. We have reliable ground truth for the eval lab
-- 3. Extraction exercises are meaningful (not just extracting nulls)

CREATE OR REPLACE TABLE CLAIMS_SAMPLE AS
SELECT
    ROW_NUMBER() OVER (ORDER BY RANDOM()) AS claim_id,
    *
FROM (
    SELECT *
    FROM RAW_CLAIMS
    WHERE building_damage_amount > 0
      AND contents_damage_amount > 0
      AND flood_event IS NOT NULL
      AND water_depth IS NOT NULL
      AND state IS NOT NULL
      AND flood_zone IS NOT NULL
    LIMIT 50
);

SELECT COUNT(*) AS sample_size FROM CLAIMS_SAMPLE;
SELECT * FROM CLAIMS_SAMPLE LIMIT 3;

---
# Section 2: Designing Extraction Schemas with Pydantic

Before building the extraction pipeline, we need to define *what* we want to extract. We'll use **Pydantic models** to:

1. Define the target schema with field types and descriptions
2. Generate a JSON schema that Cortex can use for **structured output** (guaranteeing valid JSON)
3. Validate extracted data and catch errors automatically

> **Snowflake Feature**: Cortex's `response_format` parameter accepts a JSON schema, forcing the LLM to return data that matches your exact structure. This eliminates parsing failures and dramatically reduces post-processing.

In [ ]:
import json
import time
from typing import TypedDict, Literal, Optional
from datetime import datetime

from pydantic import BaseModel, Field, field_validator
from langgraph.graph import StateGraph, END
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
# Define the extraction target schema.
# Each field has a type and description -- the descriptions help the LLM 
# understand what to look for in the unstructured text.

class ClaimExtraction(BaseModel):
    """Structured data extracted from insurance adjuster field notes."""
    
    claim_id: Optional[str] = Field(None, description="The claim identifier or number")
    date_of_loss: Optional[str] = Field(None, description="Date when the flood damage occurred (YYYY-MM-DD format)")
    state: Optional[str] = Field(None, description="Two-letter US state code (e.g., TX, FL, LA)")
    county_code: Optional[str] = Field(None, description="County FIPS code")
    flood_zone: Optional[str] = Field(None, description="FEMA flood zone designation (e.g., AE, V, X, A)")
    flood_event: Optional[str] = Field(None, description="Name of the flood event or disaster")
    water_depth_inches: Optional[int] = Field(None, description="Depth of flood water in inches")
    flood_duration_hours: Optional[int] = Field(None, description="How long flood water remained, in hours")
    building_damage_amount: Optional[float] = Field(None, description="Dollar amount of building/structural damage")
    contents_damage_amount: Optional[float] = Field(None, description="Dollar amount of contents/personal property damage")
    building_property_value: Optional[float] = Field(None, description="Assessed or estimated value of the building")
    contents_property_value: Optional[float] = Field(None, description="Assessed or estimated value of contents")
    basement_type: Optional[int] = Field(None, description="Basement/enclosure/crawlspace type code (0-4)")
    floodproofed: Optional[bool] = Field(None, description="Whether the building has floodproofing measures")

# Generate the JSON schema that Cortex will use for structured output
extraction_schema = ClaimExtraction.model_json_schema()
print("Extraction schema for Cortex structured output:")
print(json.dumps(extraction_schema, indent=2))

In [ ]:
# Test the schema with a single extraction call using Cortex structured output.
# The response_format parameter forces the LLM to return valid JSON matching our schema.

sample_note = session.sql("SELECT adjuster_notes FROM ADJUSTER_NOTES LIMIT 1").collect()[0][0]
print("Input note (first 300 chars):")
print(sample_note[:300] + "...\n")

extraction_prompt = [
    {
        "role": "system", 
        "content": "Extract structured insurance claim data from the adjuster notes. Return all fields you can find. Use null for fields not mentioned in the notes."
    },
    {"role": "user", "content": sample_note}
]

# Use Snowpark SQL to call CORTEX.COMPLETE with structured output
result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        {json.dumps(extraction_prompt)},
        {{
            'temperature': 0,
            'max_tokens': 2048,
            'response_format': {{
                'type': 'json',
                'schema': {json.dumps(extraction_schema)}
            }}
        }}
    ):choices[0]:messages::STRING AS extraction
""").collect()[0][0]

extracted = json.loads(result)
print("Extracted data:")
print(json.dumps(extracted, indent=2))

# Validate with Pydantic
claim = ClaimExtraction(**extracted)
print(f"\nPydantic validation passed! Claim ID: {claim.claim_id}")

### EXERCISE: Add Validation Rules

The basic schema above accepts any value that matches the type. But we know things about our domain — states are 2-letter codes, water depth can't be negative, etc. Add Pydantic **field validators** to catch common extraction errors.

Your validators should check:
1. `state` must be exactly 2 uppercase letters
2. `water_depth_inches` must be >= 0 and <= 600 (50 feet max)
3. `building_damage_amount` and `contents_damage_amount` must be >= 0
4. `basement_type` must be between 0 and 4

In [ ]:
# EXERCISE: Add field validators to catch extraction errors.
# 
# Example validator:
#   @field_validator('state')
#   @classmethod
#   def validate_state(cls, v):
#       if v is not None and (len(v) != 2 or not v.isalpha()):
#           raise ValueError(f"State must be 2 letters, got: {v}")
#       return v.upper() if v else v

class ClaimExtractionValidated(BaseModel):
    """Extraction schema with domain-specific validation rules."""
    
    claim_id: Optional[str] = Field(None, description="The claim identifier or number")
    date_of_loss: Optional[str] = Field(None, description="Date when the flood damage occurred (YYYY-MM-DD format)")
    state: Optional[str] = Field(None, description="Two-letter US state code")
    county_code: Optional[str] = Field(None, description="County FIPS code")
    flood_zone: Optional[str] = Field(None, description="FEMA flood zone designation (e.g., AE, V, X, A)")
    flood_event: Optional[str] = Field(None, description="Name of the flood event or disaster")
    water_depth_inches: Optional[int] = Field(None, description="Depth of flood water in inches")
    flood_duration_hours: Optional[int] = Field(None, description="How long flood water remained, in hours")
    building_damage_amount: Optional[float] = Field(None, description="Dollar amount of building/structural damage")
    contents_damage_amount: Optional[float] = Field(None, description="Dollar amount of contents/personal property damage")
    building_property_value: Optional[float] = Field(None, description="Assessed or estimated value of the building")
    contents_property_value: Optional[float] = Field(None, description="Assessed or estimated value of contents")
    basement_type: Optional[int] = Field(None, description="Basement/enclosure/crawlspace type code (0-4)")
    floodproofed: Optional[bool] = Field(None, description="Whether the building has floodproofing measures")

    # YOUR VALIDATORS HERE
    # Add @field_validator decorators for: state, water_depth_inches, 
    # building_damage_amount, contents_damage_amount, basement_type


# Test your validators -- this should pass:
valid_data = {"state": "TX", "water_depth_inches": 24, "building_damage_amount": 50000}
print("Valid data test:", ClaimExtractionValidated(**valid_data))

# This should raise a validation error (try uncommenting):
# invalid_data = {"state": "Texas", "water_depth_inches": -5}
# ClaimExtractionValidated(**invalid_data)

These Pydantic models serve double duty in our LangGraph pipeline:
- The **JSON schema** feeds into Cortex's `response_format` to constrain the LLM's output structure
- The **validators** run in the VALIDATE node to catch semantic errors the LLM might make
- When validation fails, the **error messages** feed into the FIX node to tell the LLM exactly what to correct

This is the key insight: Pydantic gives us a single source of truth for both schema enforcement and business rule validation.

---
# Section 5: Building the LangGraph Extraction Pipeline

Now we put it all together. **LangGraph** lets us build a stateful graph where each node performs a step in our extraction process, and edges define the flow — including conditional routing for retries.

Here's the graph we'll build:

```
                ┌──────────┐
                │  START   │
                └────┬─────┘
                     │
                ┌────v─────┐
                │ EXTRACT  │  Cortex COMPLETE with structured output
                └────┬─────┘
                     │
                ┌────v─────┐
                │ VALIDATE │  Pydantic model validation
                └────┬─────┘
                     │
              ┌──────┴──────┐
              │             │
          (valid)      (invalid)
              │             │
        ┌─────v────┐  ┌────v────┐
        │ FINALIZE │  │   FIX   │  Re-prompt with error details
        └─────┬────┘  └────┬────┘
              │             │
              │        ┌────v─────┐
              │        │ VALIDATE │  Re-validate (max 2 retries)
              │        └────┬─────┘
              │             │
              │      ┌──────┴──────┐
              │   (valid)    (exhausted)
              │      │             │
              │  ┌───v───┐  ┌─────v────┐
              │  │FINALIZE│  │ FINALIZE │  (with error flags)
              │  └───┬───┘  └─────┬────┘
              │      │            │
              └──────┴─────┬──────┘
                           │
                      ┌────v────┐
                      │   END   │
                      └─────────┘
```

Key LangGraph concepts:
- **State**: A `TypedDict` that flows through all nodes, accumulating data
- **Nodes**: Functions that read state and return updates
- **Edges**: Define which node runs next (can be conditional)
- **Conditional edges**: Route based on state values (our validation result)

In [ ]:
# Step 1: Define the graph state.
# This TypedDict defines ALL the data that flows through the pipeline.

class ExtractionState(TypedDict):
    # --- Inputs (set once at invocation) ---
    claim_id: str                    # Unique identifier for tracking
    adjuster_notes: str              # Raw unstructured text to extract from
    
    # --- Extraction lifecycle (updated by nodes) ---
    raw_extraction: dict             # Latest raw JSON output from Cortex
    validated_extraction: dict       # Output after successful Pydantic validation
    extraction_errors: list          # Structured validation errors
    
    # --- Control flow ---
    retry_count: int                 # Number of fix attempts so far
    max_retries: int                 # Maximum allowed retries (set at invocation)
    is_valid: bool                   # Whether latest extraction passed validation
    
    # --- Observability (for cost tracking & debugging) ---
    node_history: list               # Ordered log of visited nodes
    cortex_calls: int                # Total LLM invocations (for cost tracking)
    processing_time: float           # Cumulative wall-clock seconds

Notice the observability fields: `node_history`, `cortex_calls`, and `processing_time`. These aren't needed for the extraction logic itself, but they're critical for:
- **Cost tracking**: knowing how many LLM calls each claim required
- **Debugging**: seeing the exact path through the graph
- **Performance analysis**: measuring wall-clock time per claim

This is a best practice — instrument your pipelines from the start, not as an afterthought.

In [ ]:
# Step 2: Implement the EXTRACT node.
# This is where we call Cortex to pull structured data from the unstructured text.

def extract_node(state: ExtractionState) -> dict:
    """Extract structured data from adjuster notes using Cortex COMPLETE."""
    start_time = time.time()
    
    # Build the extraction prompt
    extraction_prompt = json.dumps([
        {
            "role": "system", 
            "content": (
                "You are an expert insurance data extraction system. "
                "Extract all structured claim information from the adjuster's field notes. "
                "Be precise with numbers, dates, and codes. "
                "If a field is not mentioned in the notes, use null. "
                "Pay special attention to: "
                "- Dollar amounts (distinguish building vs contents damage) "
                "- Geographic information (state code, county) "
                "- Flood zone designations "
                "- Dates in YYYY-MM-DD format "
                "- Numeric codes (basement type should be an integer 0-4)"
            )
        },
        {"role": "user", "content": state["adjuster_notes"]}
    ])
    
    schema_json = json.dumps(ClaimExtractionValidated.model_json_schema())
    
    # Tag this query for cost tracking
    session.sql(
        "ALTER SESSION SET QUERY_TAG = "
        "'{\"lab\": \"claims-extraction\", \"section\": \"extract\", \"node\": \"extract\"}'"
    ).collect()
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON($${extraction_prompt}$$),
            {{
                'temperature': 0,
                'max_tokens': 2048,
                'response_format': {{
                    'type': 'json',
                    'schema': PARSE_JSON($${schema_json}$$)
                }}
            }}
        ):choices[0]:messages::STRING AS result
    """).collect()[0][0]
    
    extracted = json.loads(result)
    elapsed = time.time() - start_time
    
    return {
        "raw_extraction": extracted,
        "cortex_calls": state.get("cortex_calls", 0) + 1,
        "node_history": state.get("node_history", []) + ["extract"],
        "processing_time": state.get("processing_time", 0) + elapsed
    }

In [ ]:
# Step 3: Implement the VALIDATE node.
# This node runs Pydantic validation -- no LLM calls, just business rules.

def validate_node(state: ExtractionState) -> dict:
    """Validate extracted data using the Pydantic model with domain rules."""
    errors = []
    raw = state["raw_extraction"]
    
    try:
        validated = ClaimExtractionValidated(**raw)
        return {
            "validated_extraction": validated.model_dump(),
            "extraction_errors": [],
            "is_valid": True,
            "node_history": state.get("node_history", []) + ["validate_pass"]
        }
    except Exception as e:
        # Parse Pydantic validation errors into a structured list
        # that we can feed back to the LLM in the FIX node
        if hasattr(e, 'errors'):
            for err in e.errors():
                errors.append({
                    "field": ".".join(str(x) for x in err["loc"]),
                    "message": err["msg"],
                    "type": err["type"],
                    "input_value": str(err.get("input", ""))[:100]
                })
        else:
            errors.append({"field": "unknown", "message": str(e), "type": "general"})
        
        return {
            "extraction_errors": errors,
            "is_valid": False,
            "node_history": state.get("node_history", []) + ["validate_fail"]
        }

In [ ]:
# Step 4: Implement the FIX node.
# When validation fails, we re-prompt the LLM with the specific errors.
# This is much more effective than just retrying blindly.

def fix_node(state: ExtractionState) -> dict:
    """Re-prompt Cortex with specific validation errors to fix the extraction."""
    start_time = time.time()
    
    # Format the errors into a clear description for the LLM
    error_descriptions = "\n".join([
        f"- Field '{e['field']}': {e['message']} (current value: {e['input_value']})"
        for e in state["extraction_errors"]
    ])
    
    fix_prompt = json.dumps([
        {
            "role": "system", 
            "content": (
                "You are an expert insurance data extraction system. "
                "A previous extraction attempt had validation errors. "
                "Fix ONLY the fields with errors. "
                "Keep all correctly extracted fields unchanged."
            )
        },
        {
            "role": "user", 
            "content": (
                f"Original adjuster notes:\n{state['adjuster_notes']}\n\n"
                f"Previous extraction (with errors):\n{json.dumps(state['raw_extraction'], indent=2)}\n\n"
                f"Validation errors to fix:\n{error_descriptions}\n\n"
                "Please provide the corrected extraction."
            )
        }
    ])
    
    schema_json = json.dumps(ClaimExtractionValidated.model_json_schema())
    
    session.sql(
        "ALTER SESSION SET QUERY_TAG = "
        "'{\"lab\": \"claims-extraction\", \"section\": \"extract\", \"node\": \"fix\"}'"
    ).collect()
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON($${fix_prompt}$$),
            {{
                'temperature': 0,
                'max_tokens': 2048,
                'response_format': {{
                    'type': 'json',
                    'schema': PARSE_JSON($${schema_json}$$)
                }}
            }}
        ):choices[0]:messages::STRING AS result
    """).collect()[0][0]
    
    extracted = json.loads(result)
    elapsed = time.time() - start_time
    
    return {
        "raw_extraction": extracted,
        "retry_count": state.get("retry_count", 0) + 1,
        "cortex_calls": state.get("cortex_calls", 0) + 1,
        "node_history": state.get("node_history", []) + ["fix"],
        "processing_time": state.get("processing_time", 0) + elapsed
    }

In [ ]:
# Step 5: Implement the FINALIZE node.
# Packages the result with metadata for downstream analysis.

def finalize_node(state: ExtractionState) -> dict:
    """Finalize the extraction result with metadata."""
    result = state.get("validated_extraction") or state.get("raw_extraction", {})
    
    # Attach processing metadata to the result
    result["_metadata"] = {
        "claim_id": state["claim_id"],
        "is_valid": state.get("is_valid", False),
        "retry_count": state.get("retry_count", 0),
        "cortex_calls": state.get("cortex_calls", 0),
        "node_path": " -> ".join(state.get("node_history", []) + ["finalize"]),
        "processing_time_seconds": round(state.get("processing_time", 0), 2),
        "had_errors": len(state.get("extraction_errors", [])) > 0
    }
    
    return {
        "validated_extraction": result,
        "node_history": state.get("node_history", []) + ["finalize"]
    }

In [ ]:
# Step 6: Define the routing function and assemble the graph.

def should_retry(state: ExtractionState) -> Literal["fix", "finalize"]:
    """Decide whether to retry extraction or finalize with what we have."""
    if state.get("is_valid", False):
        return "finalize"
    elif state.get("retry_count", 0) < state.get("max_retries", 2):
        return "fix"
    else:
        # Max retries exhausted -- finalize with whatever we have
        return "finalize"

# Build the graph
workflow = StateGraph(ExtractionState)

# Add nodes
workflow.add_node("extract", extract_node)
workflow.add_node("validate", validate_node)
workflow.add_node("fix", fix_node)
workflow.add_node("finalize", finalize_node)

# Add edges
workflow.set_entry_point("extract")
workflow.add_edge("extract", "validate")
workflow.add_conditional_edges(
    "validate",
    should_retry,
    {
        "fix": "fix",
        "finalize": "finalize"
    }
)
workflow.add_edge("fix", "validate")    # After fix, re-validate (creates the retry loop)
workflow.add_edge("finalize", END)

# Compile the graph
extraction_graph = workflow.compile()

print("LangGraph extraction pipeline compiled successfully!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print(f"Entry point: extract")
print(f"Conditional routing: validate -> fix (if invalid) or finalize (if valid)")

In [ ]:
# Step 7: Test the graph with a single claim!
test_claim = session.sql(
    "SELECT claim_id, adjuster_notes FROM ADJUSTER_NOTES LIMIT 1"
).collect()[0]

initial_state = {
    "claim_id": str(test_claim[0]),
    "adjuster_notes": test_claim[1],
    "raw_extraction": {},
    "validated_extraction": {},
    "extraction_errors": [],
    "retry_count": 0,
    "max_retries": 2,
    "is_valid": False,
    "node_history": [],
    "cortex_calls": 0,
    "processing_time": 0.0
}

print(f"Processing claim {initial_state['claim_id']}...")
print(f"Note preview: {initial_state['adjuster_notes'][:200]}...\n")

result = extraction_graph.invoke(initial_state)

print(f"Node path: {' -> '.join(result['node_history'])}")
print(f"Valid: {result['is_valid']}")
print(f"Cortex calls: {result['cortex_calls']}")
print(f"Processing time: {result['processing_time']:.2f}s")
print(f"Retries needed: {result['retry_count']}")
print(f"\nExtracted data:")
print(json.dumps(result['validated_extraction'], indent=2))

### CHALLENGE: Add a Confidence Scoring Node

The current pipeline extracts and validates, but doesn't tell us *how confident* we should be in the results. Add a `confidence_score` node between VALIDATE and FINALIZE that:

1. Counts how many fields were extracted (non-null count out of 14 total fields)
2. Assigns a confidence level:
   - **high**: > 80% of fields extracted AND validation passed
   - **medium**: 50-80% of fields extracted
   - **low**: < 50% of fields extracted
3. Adds a `confidence` field to the state

**Steps**:
1. Define a `confidence_score_node(state)` function
2. Rebuild the graph with the new node inserted after validate
3. Test it with a sample claim

In [ ]:
# CHALLENGE: Implement your confidence scoring node here.

# def confidence_score_node(state: ExtractionState) -> dict:
#     """Score confidence based on extraction completeness."""
#     extraction = state.get("validated_extraction") or state.get("raw_extraction", {})
#     
#     # Count non-null fields (excluding _metadata)
#     total_fields = 14
#     extracted_fields = sum(1 for k, v in extraction.items() 
#                          if v is not None and k != "_metadata")
#     completeness = extracted_fields / total_fields
#     
#     # YOUR LOGIC HERE: assign confidence based on completeness and validity
#     
#     return {...}

# Then rebuild the graph with your new node:
# workflow2 = StateGraph(ExtractionState)
# ... add all nodes including your new one ...
# extraction_graph = workflow2.compile()

### Section 5 Recap

We built a complete LangGraph extraction pipeline with:

- **Automatic retry on validation failure** — the FIX node re-prompts with specific error details
- **Structured error feedback** — Pydantic errors tell the LLM exactly what's wrong
- **Full observability** — every node logs its activity, call count, and timing
- **Clean architecture** — each node has a single responsibility, making the pipeline easy to extend (as you saw in the challenge)

Next, we'll run this pipeline across all our claims in batch.

---
# Section 6: Running Extraction at Scale

Processing a single claim is straightforward. Processing hundreds or thousands requires careful engineering:

- **Progress tracking** — know what's been processed so you can resume after interruptions
- **Error isolation** — one bad claim shouldn't kill the entire batch
- **Result persistence** — save results to Snowflake tables, not just Python variables
- **Idempotency** — re-running a cell shouldn't create duplicates

> **Snowflake Feature**: We'll use `MERGE INTO` for idempotent upserts and the `VARIANT` data type to store our flexible JSON extraction results alongside metadata.

In [ ]:
-- Create tables for results and progress tracking.
-- Using IF NOT EXISTS makes this safe to re-run after a kernel restart.

CREATE TABLE IF NOT EXISTS EXTRACTION_RESULTS (
    claim_id STRING,
    extracted_data VARIANT,
    is_valid BOOLEAN,
    retry_count INTEGER,
    cortex_calls INTEGER,
    processing_time_seconds FLOAT,
    node_path STRING,
    extraction_errors VARIANT,
    extracted_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS EXTRACTION_PROGRESS (
    claim_id STRING PRIMARY KEY,
    status STRING,          -- 'completed', 'failed', 'in_progress'
    started_at TIMESTAMP_NTZ,
    completed_at TIMESTAMP_NTZ
);

SELECT 'Tables ready' AS status,
    (SELECT COUNT(*) FROM EXTRACTION_RESULTS) AS existing_results,
    (SELECT COUNT(*) FROM EXTRACTION_PROGRESS) AS existing_progress;

In [ ]:
# Batch extraction function with checkpointing and error isolation.
# Key design decisions:
#   1. Query for UNPROCESSED claims (skips already-completed ones on restart)
#   2. MERGE for idempotent progress tracking (no duplicates on re-run)
#   3. try/except per claim (one failure doesn't stop the batch)
#   4. Results saved to Snowflake table after EACH claim (not just at the end)

def run_batch_extraction(batch_size=10, max_retries=2):
    """Run extraction on unprocessed claims with per-claim checkpointing."""
    
    # Find claims that haven't been processed yet -- this is the restartability key
    unprocessed = session.sql(f"""
        SELECT an.claim_id, an.adjuster_notes
        FROM ADJUSTER_NOTES an
        LEFT JOIN EXTRACTION_PROGRESS ep ON an.claim_id::STRING = ep.claim_id
        WHERE ep.claim_id IS NULL OR ep.status = 'failed'
        LIMIT {batch_size}
    """).collect()
    
    total = len(unprocessed)
    if total == 0:
        print("All claims already processed! Nothing to do.")
        return [], []
    
    print(f"Processing {total} unprocessed claims (batch_size={batch_size})...\n")
    results = []
    errors = []
    
    for i, row in enumerate(unprocessed):
        claim_id = str(row[0])
        notes = row[1]
        print(f"  [{i+1}/{total}] Claim {claim_id}...", end=" ", flush=True)
        
        # Mark as in-progress using MERGE (idempotent)
        session.sql(f"""
            MERGE INTO EXTRACTION_PROGRESS ep
            USING (SELECT '{claim_id}' AS claim_id) src
            ON ep.claim_id = src.claim_id
            WHEN MATCHED THEN UPDATE SET status='in_progress', started_at=CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (claim_id, status, started_at)
                VALUES (src.claim_id, 'in_progress', CURRENT_TIMESTAMP())
        """).collect()
        
        try:
            state = {
                "claim_id": claim_id,
                "adjuster_notes": notes,
                "raw_extraction": {},
                "validated_extraction": {},
                "extraction_errors": [],
                "retry_count": 0,
                "max_retries": max_retries,
                "is_valid": False,
                "node_history": [],
                "cortex_calls": 0,
                "processing_time": 0.0
            }
            
            result = extraction_graph.invoke(state)
            
            # Save result to table
            extracted_json = json.dumps(result['validated_extraction']).replace("'", "''")
            errors_json = json.dumps(result.get('extraction_errors', [])).replace("'", "''")
            node_path = ' -> '.join(result['node_history'])
            
            session.sql(f"""
                INSERT INTO EXTRACTION_RESULTS
                (claim_id, extracted_data, is_valid, retry_count, cortex_calls, 
                 processing_time_seconds, node_path, extraction_errors)
                SELECT
                    '{claim_id}',
                    PARSE_JSON('{extracted_json}'),
                    {result['is_valid']},
                    {result['retry_count']},
                    {result['cortex_calls']},
                    {result['processing_time']:.2f},
                    '{node_path}',
                    PARSE_JSON('{errors_json}')
            """).collect()
            
            # Mark completed
            session.sql(f"""
                UPDATE EXTRACTION_PROGRESS
                SET status='completed', completed_at=CURRENT_TIMESTAMP()
                WHERE claim_id='{claim_id}'
            """).collect()
            
            status = "VALID" if result['is_valid'] else "INVALID"
            print(f"{status} ({result['cortex_calls']} calls, {result['processing_time']:.1f}s)")
            results.append(result)
            
        except Exception as e:
            print(f"FAILED: {str(e)[:80]}")
            session.sql(f"""
                UPDATE EXTRACTION_PROGRESS
                SET status='failed', completed_at=CURRENT_TIMESTAMP()
                WHERE claim_id='{claim_id}'
            """).collect()
            errors.append({"claim_id": claim_id, "error": str(e)})
    
    print(f"\nBatch complete: {len(results)} succeeded, {len(errors)} failed")
    return results, errors

In [ ]:
# Run a small initial batch to verify everything works end-to-end
results, errors = run_batch_extraction(batch_size=10, max_retries=2)

### EXPERIMENT: Batch Size

Try running with `batch_size=20` in the cell below. Does it take roughly twice as long as the batch of 10? Why or why not?

Consider: each claim is processed **sequentially** in our Python loop. What would happen if we could parallelize? (We'll explore this in the Performance section.)

In [ ]:
# Process the remaining claims. Adjust batch_size based on your time budget.
# Because of restartability, this picks up where the previous batch left off.
results, errors = run_batch_extraction(batch_size=100, max_retries=2)

In [ ]:
-- Check overall extraction progress
SELECT
    status,
    COUNT(*) AS claim_count,
    ROUND(AVG(DATEDIFF('second', started_at, completed_at)), 1) AS avg_duration_seconds
FROM EXTRACTION_PROGRESS
GROUP BY status
ORDER BY claim_count DESC;

In [ ]:
-- Preview extraction results using Snowflake's VARIANT : notation
-- This is a powerful feature for querying semi-structured data stored as JSON
SELECT
    claim_id,
    is_valid,
    retry_count,
    cortex_calls,
    ROUND(processing_time_seconds, 1) AS time_sec,
    node_path,
    extracted_data:state::STRING AS extracted_state,
    extracted_data:building_damage_amount::FLOAT AS extracted_bldg_damage,
    extracted_data:water_depth_inches::INTEGER AS extracted_water_depth,
    extracted_data:flood_zone::STRING AS extracted_flood_zone
FROM EXTRACTION_RESULTS
ORDER BY extracted_at DESC
LIMIT 10;

---
# Section 7: Quick Accuracy Check — Eval Lab Preview

Because we generated our adjuster notes from structured data, we have **ground truth** — we know exactly what the correct extracted values should be. Let's do a quick accuracy check by comparing extracted values to the originals.

> This is a preview. The next lab will build a comprehensive evaluation framework with precision/recall, confusion matrices, and regression metrics.

In [ ]:
-- Join extracted results to ground truth and compute field-level accuracy.
-- For text fields: exact match. For numbers: within 1% tolerance.
CREATE OR REPLACE VIEW EXTRACTION_ACCURACY AS
SELECT
    er.claim_id,
    er.is_valid,
    er.retry_count,
    er.cortex_calls,
    -- Exact match checks (text/categorical fields)
    CASE WHEN er.extracted_data:state::STRING = cs.state THEN 1 ELSE 0 END AS state_match,
    CASE WHEN er.extracted_data:flood_zone::STRING = cs.flood_zone THEN 1 ELSE 0 END AS flood_zone_match,
    CASE WHEN er.extracted_data:flood_event::STRING = cs.flood_event THEN 1 ELSE 0 END AS flood_event_match,
    CASE WHEN er.extracted_data:water_depth_inches::INTEGER = cs.water_depth THEN 1 ELSE 0 END AS water_depth_match,
    CASE WHEN er.extracted_data:basement_type::INTEGER = cs.basement_type THEN 1 ELSE 0 END AS basement_type_match,
    -- Numeric proximity checks (within 1% tolerance for dollar amounts)
    CASE WHEN ABS(er.extracted_data:building_damage_amount::FLOAT - cs.building_damage_amount)
         / NULLIF(cs.building_damage_amount, 0) < 0.01 THEN 1 ELSE 0 END AS building_damage_match,
    CASE WHEN ABS(er.extracted_data:contents_damage_amount::FLOAT - cs.contents_damage_amount)
         / NULLIF(cs.contents_damage_amount, 0) < 0.01 THEN 1 ELSE 0 END AS contents_damage_match,
    CASE WHEN ABS(er.extracted_data:building_property_value::FLOAT - cs.building_property_value)
         / NULLIF(cs.building_property_value, 0) < 0.01 THEN 1 ELSE 0 END AS property_value_match
FROM EXTRACTION_RESULTS er
JOIN CLAIMS_SAMPLE cs ON er.claim_id = cs.claim_id::STRING;

In [ ]:
-- Aggregate accuracy across all processed claims
SELECT
    COUNT(*) AS total_claims,
    SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) AS valid_extractions,
    ROUND(AVG(state_match) * 100, 1) AS state_accuracy_pct,
    ROUND(AVG(flood_zone_match) * 100, 1) AS flood_zone_accuracy_pct,
    ROUND(AVG(flood_event_match) * 100, 1) AS flood_event_accuracy_pct,
    ROUND(AVG(water_depth_match) * 100, 1) AS water_depth_accuracy_pct,
    ROUND(AVG(basement_type_match) * 100, 1) AS basement_type_accuracy_pct,
    ROUND(AVG(building_damage_match) * 100, 1) AS building_damage_accuracy_pct,
    ROUND(AVG(contents_damage_match) * 100, 1) AS contents_damage_accuracy_pct,
    ROUND(AVG(property_value_match) * 100, 1) AS property_value_accuracy_pct,
    ROUND(AVG(retry_count), 2) AS avg_retries,
    ROUND(AVG(cortex_calls), 2) AS avg_cortex_calls
FROM EXTRACTION_ACCURACY;

### EXERCISE: Which Fields Are Hardest to Extract?

Write a query that shows per-field accuracy ranked from worst to best. Then answer:
- Why might some fields be harder to extract than others?
- Does the accuracy differ for claims that required retries vs those that didn't?
- What could you change in the extraction prompt to improve the worst-performing fields?

In [ ]:
-- EXERCISE: Write your accuracy analysis queries here.
-- 
-- Hint: You can UNPIVOT the accuracy columns to compare fields side by side:
-- SELECT field_name, accuracy FROM (
--     SELECT 'state' AS field_name, AVG(state_match)*100 AS accuracy FROM EXTRACTION_ACCURACY
--     UNION ALL
--     SELECT 'flood_zone', AVG(flood_zone_match)*100 FROM EXTRACTION_ACCURACY
--     ...
-- ) ORDER BY accuracy ASC;
-- 
-- Bonus: Compare accuracy WHERE retry_count = 0 vs retry_count > 0

-- YOUR CODE HERE


### Coming Up: The Eval Lab

This was a quick accuracy check using exact-match and tolerance-based comparisons. In the **next lab (Model Evaluation)**, we'll build a comprehensive evaluation framework covering:

- **Precision and recall** per field
- **Confusion matrices** for categorical fields (state, flood zone)
- **Regression metrics** (MAE, MAPE) for numeric fields
- **Semantic similarity** for text fields (flood event names)
- **Error analysis** — systematic patterns in what the model gets wrong

The tables you created here — `EXTRACTION_RESULTS` and `CLAIMS_SAMPLE` — are the inputs to that lab. Don't drop them!

---
# Section 8: Cost Analysis

Understanding and controlling LLM costs is a critical production skill. Throughout this lab, we've been tagging every query with structured JSON metadata. Now we'll use Snowflake's `QUERY_HISTORY` to analyze exactly where our credits went.

> **Snowflake Feature**: `INFORMATION_SCHEMA.QUERY_HISTORY` provides near-real-time access to query execution metadata including credits consumed, elapsed time, and your custom `QUERY_TAG`. Combined with structured JSON tags, this gives you fine-grained cost attribution without any external tooling.

In [ ]:
-- Cost breakdown by lab section.
-- Our JSON query tags let us slice costs by any dimension we tagged.
SELECT
    PARSE_JSON(query_tag):section::STRING AS lab_section,
    COUNT(*) AS query_count,
    ROUND(SUM(credits_used_cloud_services), 4) AS total_credits,
    ROUND(SUM(total_elapsed_time) / 1000, 1) AS total_time_seconds,
    ROUND(AVG(total_elapsed_time) / 1000, 1) AS avg_time_seconds
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    DATE_RANGE_START => DATEADD('hour', -6, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10000
))
WHERE query_tag LIKE '%claims-extraction%'
  AND query_tag IS NOT NULL
GROUP BY lab_section
ORDER BY total_credits DESC;

In [ ]:
-- Drill into extraction costs by LangGraph node.
-- This shows us the cost of extract vs fix (retry) operations.
SELECT
    PARSE_JSON(query_tag):node::STRING AS graph_node,
    COUNT(*) AS call_count,
    ROUND(SUM(credits_used_cloud_services), 4) AS total_credits,
    ROUND(AVG(total_elapsed_time) / 1000, 2) AS avg_time_seconds
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    DATE_RANGE_START => DATEADD('hour', -6, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10000
))
WHERE query_tag LIKE '%claims-extraction%'
  AND query_tag LIKE '%"node"%'
  AND query_text ILIKE '%CORTEX%COMPLETE%'
GROUP BY graph_node
ORDER BY total_credits DESC;

In [ ]:
-- Calculate cost per claim and project to the full dataset.
SELECT
    (SELECT COUNT(*) FROM EXTRACTION_RESULTS) AS claims_processed,
    ROUND(SUM(credits_used_cloud_services), 4) AS total_extraction_credits,
    ROUND(
        SUM(credits_used_cloud_services) / NULLIF((SELECT COUNT(*) FROM EXTRACTION_RESULTS), 0), 
        6
    ) AS credits_per_claim,
    -- Project to full dataset (2.7M claims)
    ROUND(
        SUM(credits_used_cloud_services) / NULLIF((SELECT COUNT(*) FROM EXTRACTION_RESULTS), 0) * 2700000, 
        2
    ) AS projected_credits_full_dataset
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    DATE_RANGE_START => DATEADD('hour', -6, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10000
))
WHERE query_tag LIKE '%"section": "extract"%'
  AND query_text ILIKE '%CORTEX%COMPLETE%';

### CHALLENGE: Measure the Cost of Retries

Retries improve quality but cost extra credits. Quantify the trade-off:

1. What **percentage** of total extraction credits were spent on fix/retry operations?
2. What is the **average cost per claim** for claims with 0, 1, and 2 retries?
3. If you could **eliminate all retries** (by improving the prompt or schema), how much would you save?
4. Given your cost-per-claim, what's the **projected cost** to process all 2.7M claims?

In [ ]:
-- CHALLENGE: Write your cost analysis queries here.
-- 
-- Hint for question 2: Join EXTRACTION_RESULTS (which has retry_count per claim)
-- with cost data from QUERY_HISTORY
-- 
-- Hint for question 3: The fix node queries have tag containing "node": "fix"

-- YOUR CODE HERE


### Cost Takeaways

- **`temperature=0`** produces more consistent outputs, leading to fewer validation failures and fewer costly retries
- **Structured output schemas** (`response_format`) eliminate JSON parsing failures entirely — a huge cost saver at scale
- **Query tags** are cheap insurance — they cost nothing to set but give you complete cost visibility
- **The retry pattern trades cost for quality** — the right balance depends on your accuracy requirements and budget

---
# Section 9: Performance Optimization

Our Python-loop approach processes claims sequentially. But Snowflake has a powerful trick: when you call `CORTEX.COMPLETE` in a SQL `SELECT` across multiple rows, **Snowflake can parallelize those calls automatically**.

In this section, we'll compare three approaches:
1. **Python loop** (sequential, with LangGraph validation)
2. **SQL batch** (parallel, without validation)
3. **Hybrid** (SQL batch extract + Python validate + LangGraph fix only failures)

> **Snowflake Feature**: Warehouse sizing directly impacts how many Cortex calls can run in parallel. A larger warehouse means more compute nodes, which means more concurrent LLM calls.

In [ ]:
-- EXPERIMENT: SQL batch extraction (Snowflake parallelizes across rows).
-- Compare the wall-clock time of this single SQL statement processing 5 claims
-- vs the Python loop processing 5 claims above.

ALTER SESSION SET QUERY_TAG = '{"lab": "claims-extraction", "section": "perf", "experiment": "sql-batch"}';

SELECT
    claim_id,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        [
            {'role': 'system', 'content': 'Extract structured insurance claim data from adjuster notes. Return JSON with these fields: claim_id, date_of_loss, state, county_code, flood_zone, flood_event, water_depth_inches, flood_duration_hours, building_damage_amount, contents_damage_amount, building_property_value, contents_property_value, basement_type, floodproofed. Use null for missing fields.'},
            {'role': 'user', 'content': adjuster_notes}
        ],
        {'temperature': 0, 'max_tokens': 2048}
    ):choices[0]:messages::STRING AS raw_extraction
FROM ADJUSTER_NOTES
LIMIT 5;

### EXPERIMENT: Warehouse Sizing Impact

Let's test whether a larger warehouse lets Snowflake parallelize more Cortex calls. Run the same extraction on different warehouse sizes and compare wall-clock time.

In [ ]:
# EXPERIMENT: Compare extraction throughput at different warehouse sizes.
# We use a consistent 10-claim batch so the comparison is fair.

import time

extraction_sql = """
    SELECT claim_id,
        SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            [
                {{'role': 'system', 'content': 'Extract structured insurance claim data. Return JSON with: claim_id, date_of_loss, state, flood_zone, flood_event, water_depth_inches, building_damage_amount, contents_damage_amount, basement_type, floodproofed. Use null for missing fields.'}},
                {{'role': 'user', 'content': adjuster_notes}}
            ],
            {{'temperature': 0, 'max_tokens': 2048}}
        ):choices[0]:messages::STRING AS extraction
    FROM ADJUSTER_NOTES
    LIMIT 10
"""

results = {}
for size in ['XSMALL', 'SMALL']:
    session.sql(f"ALTER WAREHOUSE CLAIMS_LAB_WH SET WAREHOUSE_SIZE = '{size}'").collect()
    session.sql(f"ALTER SESSION SET QUERY_TAG = '{{\"lab\": \"claims-extraction\", \"section\": \"perf\", \"experiment\": \"{size.lower()}-batch\"}}'").collect()
    
    start = time.time()
    session.sql(extraction_sql).collect()
    elapsed = time.time() - start
    
    results[size] = elapsed
    print(f"{size:>8}: {elapsed:.1f}s for 10 claims ({10/elapsed:.2f} claims/sec)")

# Compare
if results.get('SMALL') and results.get('XSMALL'):
    speedup = results['XSMALL'] / results['SMALL']
    print(f"\nSMALL is {speedup:.1f}x faster than XSMALL")
    print(f"At SMALL throughput, 2.7M claims would take: {2700000 / (10/results['SMALL']) / 3600:.1f} hours")

# Reset to XSMALL
session.sql("ALTER WAREHOUSE CLAIMS_LAB_WH SET WAREHOUSE_SIZE = 'XSMALL'").collect()

In [ ]:
# HYBRID APPROACH: SQL batch extract → Python batch validate → LangGraph fix only failures
# This gives us the speed of SQL parallelism AND the quality of LangGraph validation.

start_hybrid = time.time()

# Step 1: Bulk extract via SQL (fast, parallel)
raw_results = session.sql("""
    SELECT claim_id, adjuster_notes,
        SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            [
                {'role': 'system', 'content': 'Extract structured insurance claim data. Return JSON with: claim_id, date_of_loss, state, county_code, flood_zone, flood_event, water_depth_inches, flood_duration_hours, building_damage_amount, contents_damage_amount, building_property_value, contents_property_value, basement_type, floodproofed. Use null for missing.'},
                {'role': 'user', 'content': adjuster_notes}
            ],
            {'temperature': 0, 'max_tokens': 2048}
        ):choices[0]:messages::STRING AS raw_extraction
    FROM ADJUSTER_NOTES
    LIMIT 10
""").to_pandas()
sql_time = time.time() - start_hybrid

# Step 2: Validate each result with Pydantic (fast, no LLM calls)
valid_count = 0
needs_fix = []
for _, row in raw_results.iterrows():
    try:
        extracted = json.loads(row['RAW_EXTRACTION'])
        ClaimExtractionValidated(**extracted)
        valid_count += 1
    except Exception:
        needs_fix.append(row)

validate_time = time.time() - start_hybrid - sql_time

# Step 3: Only send FAILED ones through LangGraph fix loop
fixed_count = 0
for row in needs_fix:
    try:
        state = {
            "claim_id": str(row['CLAIM_ID']),
            "adjuster_notes": row['ADJUSTER_NOTES'],
            "raw_extraction": json.loads(row['RAW_EXTRACTION']),
            "validated_extraction": {},
            "extraction_errors": [],
            "retry_count": 0, "max_retries": 1,
            "is_valid": False, "node_history": ["extract"],
            "cortex_calls": 1, "processing_time": 0.0
        }
        # Start from validate (skip extract since we already have raw data)
        validate_result = validate_node(state)
        state.update(validate_result)
        if not state["is_valid"]:
            fix_result = fix_node(state)
            state.update(fix_result)
        fixed_count += 1
    except Exception:
        pass

total_hybrid = time.time() - start_hybrid

print(f"HYBRID APPROACH RESULTS:")
print(f"  SQL batch extract (10 claims, parallel): {sql_time:.1f}s")
print(f"  Python validation (instant):             {validate_time:.1f}s")
print(f"  LangGraph fix ({len(needs_fix)} failures):         {total_hybrid - sql_time - validate_time:.1f}s")
print(f"  Total:                                   {total_hybrid:.1f}s")
print(f"  Valid on first try: {valid_count}/10, Needed fix: {len(needs_fix)}/10")

### Performance Comparison

| Approach | Parallelism | Validation | Retry | Best For |
|----------|------------|------------|-------|----------|
| **Python loop + LangGraph** | Sequential | Full Pydantic | Auto-retry | Small batches, maximum quality |
| **SQL batch** | Parallel (Snowflake) | None | None | Maximum throughput, simple extraction |
| **Hybrid** | Parallel extract + sequential fix | Full Pydantic | Only failures | **Production workloads** — fast AND reliable |

The hybrid approach is usually the best choice for production because:
- Most extractions pass validation on the first try (so the fix loop rarely runs)
- SQL parallelism handles the bulk of the work
- You only pay the sequential LangGraph cost for the small fraction that fails

---
# Section 10: Scaling to the Full Dataset

So far we've worked with a ~500 row sample. The full FEMA dataset has **2.7 million claims**. How do you go from lab to production scale? This isn't just about "throwing a bigger warehouse at it" — it requires a strategy that balances **cost**, **throughput**, **reliability**, and **time**.

In this section, you'll:
1. Project cost and time from your sample metrics
2. Test real throughput at different warehouse sizes
3. Explore partitioned processing for checkpointing
4. Design your own scaling plan

### EXERCISE: Project Full-Dataset Cost and Time

Using your metrics from Sections 8 and 9, calculate:
1. At your current **credits per claim**, what would it cost to process all 2.7M rows?
2. At your current **seconds per claim** (Python loop), how many hours would that take?
3. At your SQL batch throughput, how many hours would that take?
4. What's the monthly cost if you need to re-process claims as they're updated?

In [ ]:
-- EXERCISE: Calculate your projected scaling numbers.
-- Fill in the values from your earlier experiments.

WITH my_metrics AS (
    SELECT
        (SELECT COUNT(*) FROM EXTRACTION_RESULTS) AS claims_processed,
        (SELECT AVG(processing_time_seconds) FROM EXTRACTION_RESULTS) AS avg_seconds_per_claim,
        (SELECT AVG(cortex_calls) FROM EXTRACTION_RESULTS) AS avg_cortex_calls_per_claim
)
SELECT
    claims_processed,
    ROUND(avg_seconds_per_claim, 2) AS avg_sec_per_claim_python,
    ROUND(avg_cortex_calls_per_claim, 2) AS avg_cortex_calls,
    
    -- Project to 2.7M claims (Python loop approach)
    ROUND(avg_seconds_per_claim * 2700000 / 3600, 1) AS projected_hours_python_loop,
    ROUND(avg_seconds_per_claim * 2700000 / 3600 / 24, 1) AS projected_days_python_loop
    
    -- TODO: Add your SQL batch throughput projection here
    -- Hint: Use the claims/sec number from your Section 9 experiments
    
FROM my_metrics;

### Strategy 1: SQL-Native Batch Extraction (CTAS)

The simplest scaling approach: use `CREATE TABLE AS SELECT` with `CORTEX.COMPLETE` applied to every row. Snowflake handles all the parallelism for you.

In [ ]:
-- Strategy 1: SQL CTAS at scale.
-- We'll run on a larger slice (100 rows) to get a better throughput measurement.
-- In production, you'd remove the LIMIT entirely.

ALTER SESSION SET QUERY_TAG = '{"lab": "claims-extraction", "section": "scale", "experiment": "ctas-100"}';

CREATE OR REPLACE TABLE EXTRACTION_SCALE_TEST AS
SELECT
    claim_id,
    adjuster_notes,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        [
            {'role': 'system', 'content': 'Extract structured insurance claim data from adjuster notes. Return JSON with: claim_id, date_of_loss, state, county_code, flood_zone, flood_event, water_depth_inches, flood_duration_hours, building_damage_amount, contents_damage_amount, building_property_value, contents_property_value, basement_type, floodproofed. Use null for missing fields.'},
            {'role': 'user', 'content': adjuster_notes}
        ],
        {'temperature': 0, 'max_tokens': 2048}
    ):choices[0]:messages::STRING AS raw_extraction,
    CURRENT_TIMESTAMP() AS extracted_at
FROM ADJUSTER_NOTES
LIMIT 100;

-- Check the results
SELECT COUNT(*) AS rows_extracted, 
       MIN(extracted_at) AS started,
       MAX(extracted_at) AS finished,
       DATEDIFF('second', MIN(extracted_at), MAX(extracted_at)) AS elapsed_seconds
FROM EXTRACTION_SCALE_TEST;

### Strategy 2: Multi-Cluster Warehouses

For even higher throughput, Snowflake's **multi-cluster warehouses** can automatically scale out by adding clusters. Each cluster can process rows in parallel independently.

> This is different from a bigger warehouse (which has more nodes in a single cluster). Multi-cluster adds *separate clusters* that can handle concurrent queries or parts of a large query.

In [ ]:
-- Strategy 2: Configure multi-cluster warehouse for auto-scaling.
-- NOTE: This requires Enterprise Edition or higher.
-- If you're on Standard Edition, this cell will show the concept but may not execute.

-- View current warehouse config
SHOW WAREHOUSES LIKE 'CLAIMS_LAB_WH';

-- To enable multi-cluster (Enterprise Edition):
-- ALTER WAREHOUSE CLAIMS_LAB_WH SET
--     WAREHOUSE_SIZE = 'SMALL',
--     MIN_CLUSTER_COUNT = 1,
--     MAX_CLUSTER_COUNT = 3,
--     SCALING_POLICY = 'STANDARD';
--
-- With this config, Snowflake will automatically add up to 3 clusters
-- when the workload demands it, and scale back down when idle.
-- At SMALL size with 3 clusters, you get 3x the parallelism.

### Strategy 3: Partitioned Processing

For truly large datasets, partition by a natural key (state, year, flood event) and process each partition independently. This gives you:
- **Natural checkpointing** — resume at the partition level
- **Cost allocation** — attribute costs to specific partitions  
- **Parallelism** — run multiple partitions concurrently on separate warehouses or queries

In [ ]:
# Strategy 3: Partitioned processing by state.
# Each partition can be processed independently -- great for checkpointing.

# See how the data distributes across states
partition_stats = session.sql("""
    SELECT state, COUNT(*) AS claim_count
    FROM RAW_CLAIMS
    WHERE state IS NOT NULL
    GROUP BY state
    ORDER BY claim_count DESC
""").to_pandas()

print(f"Data spans {len(partition_stats)} states")
print(f"Largest partition: {partition_stats.iloc[0]['STATE']} ({partition_stats.iloc[0]['CLAIM_COUNT']:,.0f} claims)")
print(f"Smallest partition: {partition_stats.iloc[-1]['STATE']} ({partition_stats.iloc[-1]['CLAIM_COUNT']:,.0f} claims)")
print(f"\nTop 10 states (by claim volume):")
print(partition_stats.head(10).to_string(index=False))

print(f"\nWith partitioned processing, you could:")
print(f"  - Run each state as a separate CTAS query")
print(f"  - Track progress at the state level")
print(f"  - Resume by re-running only failed/incomplete states")
print(f"  - Allocate separate warehouse budgets per state/region")